<a href="https://colab.research.google.com/github/tanjun8802/Mase_EDGE/blob/jtan%2Fdev/EDGE_NAS_Search.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## EDGE — Optuna study and ExecuTorch (XNNPACK PT2E) export

Optuna searches **per-layer XNNPACK-valid static INT8** (symmetric weights, asymmetric **static** activations; per-channel or per-tensor) plus **no quantization** per `Conv2d` / `Linear`, and **L1 unstructured pruning** via `torch.nn.utils.prune`. **Dynamic** activation PT2E is omitted here because it inserts `quantized_decomposed` ops that commonly fail ExecuTorch `to_executorch()` (missing out variants) on many torch+executorch versions. Export uses `prepare_pt2e` / `convert_pt2e` and `to_edge_transform_and_lower` with `XnnpackPartitioner`. Phone metrics still come from ADB + the Image Classification demo app.

In [1]:
# # For colab use

# # !git clone --branch jtan/dev https://github.com/tanjun8802/Mase_EDGE.git 
# # %cd /content
# # !rm -rf Mase_EDGE/mase
# # %cd Mase_EDGE
# !git submodule update --init --recursive
# %cd mase/
# !pip install -e .

In [2]:
# !git submodule update --init --recursive
# %cd ./mase
# %pip install -e ".[executorch]"

In [3]:
# # Get the dataset (Tiny ImageNet)

# %cd /content/Mase_EDGE
# !wget http://cs231n.stanford.edu/tiny-imagenet-200.zip
# !unzip tiny-imagenet-200.zip

In [4]:
# pip install executorch  (and compatible torch / torchao per ExecuTorch docs)
import optuna
import torch
import torch.nn as nn
import torchvision.models as models
import torch.nn.utils.prune as prune
import torchvision.transforms as T
import subprocess
import json
import time
import warnings
from tqdm.auto import tqdm
import re
import os
import sys
from pathlib import Path
from typing import Any, Dict, List, Optional, Tuple

from torchvision import datasets

# ExecuTorch + PT2E (install: pip install executorch)
from executorch.backends.xnnpack.quantizer.xnnpack_quantizer import (
    XNNPACKQuantizer,
    get_symmetric_quantization_config,
)
from executorch.backends.xnnpack.partition.xnnpack_partitioner import XnnpackPartitioner
from executorch.exir import to_edge_transform_and_lower
from torchao.quantization.pt2e.quantize_pt2e import convert_pt2e, prepare_pt2e

# Jupyter has no __file__; assume the kernel cwd is the repo root (ADLSProject).
try:
    PROJECT_ROOT = Path(__file__).resolve().parent
except NameError:
    PROJECT_ROOT = Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from EDGE_device.device_specifications import (
    get_adb_output,
    get_adb_sh_c,
    get_hardware_for_phone,
    load_phone_specs,
)

/Users/souparna/.julia/conda/3/aarch64/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/Users/souparna/.julia/conda/3/aarch64/lib/python3.12/site-packages/torch/cuda/__init__.py:63: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]
Skipping import of cpp extensions due to incompatible torch version 2.9.1 for torchao version 0.14.0         Please see GitHub issue #2919 for more info
W0327 11:31:58.593000 80002 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


In [5]:
# Android applicationId — must match app/build.gradle.kts defaultConfig.applicationId
IMAGE_CLASSIFICATION_APP_ID = "com.image_classification_app"

# Host path to a folder with train/, val/, test/ (pushed on trial 0 only). If None, no dataset
# is sent and the app writes error metrics (acc=0, lat=999) — easy to mistake for "bad polling".
for _cand in (
    PROJECT_ROOT / "mase" / "tiny-imagenet-200",
    PROJECT_ROOT / "tiny-imagenet-200",
):
    _default_ds = _cand
    if _cand.is_dir():
        break
else:
    _default_ds = PROJECT_ROOT / "mase" / "tiny-imagenet-200"

EDGE_DEVICE_DATASET_PATH: Optional[str] = (
    str(_default_ds) if _default_ds.is_dir() else None
)
if EDGE_DEVICE_DATASET_PATH:
    print(f"EDGE_DEVICE_DATASET_PATH → {EDGE_DEVICE_DATASET_PATH}")
else:
    print(
        "WARN: No dataset path (cwd/mase/tiny-imagenet-200 missing). "
        "Trial 0 will not push data; expect acc=0 / lat=999 from the app."
    )
# Which split the app evaluates: "train" | "val" | "test"
EDGE_EVAL_SPLIT: str = "val"

# Full Tiny ImageNet val can take tens of minutes on device; override via env.
EDGE_METRICS_POLL_TIMEOUT_FLAG: int = 3600
EDGE_BENCHMARK_MAX_IMAGES_FLAG: Optional[int] = 100  # None = no cap on device eval images
BASE_STATE_DICT = None  # CPU weight snapshot after one-time train; fresh nn.Module each trial

# Base ResNet18 weights: python notebooks/resnet18_qat_fp32.pt (copy or export there)
RESNET18_PT_PATH = PROJECT_ROOT / "python notebooks" / "resnet18_qat_fp32.pt"
MOBILENET_V3_PATH = PROJECT_ROOT / "python notebooks" / "mobilenet_v3_large_qat_fp32.pt"
# PTQ/QAT training if used
EPOCHS = 1
LR = 1e-4
BATCH_SIZE = 64


def list_xnnpack_quantizable_module_names(model: nn.Module) -> List[str]:
    """Module FQNs for XNNPACK-supported weight quant (conv / linear in nn.Module tree)."""
    return [
        n
        for n, m in model.named_modules()
        if n and isinstance(m, (nn.Conv2d, nn.Linear))
    ]


# Must match the architecture used in load_model_and_train() (MobileNet V3 Large).
_tm = models.mobilenet_v3_large(weights=None)
QUANT_LAYER_NAMES: List[str] = list_xnnpack_quantizable_module_names(_tm)
del _tm


def optuna_param_name_for_module(module_fqn: str) -> str:
    """Optuna parameter name (no dots)."""
    return "q_" + re.sub(r"[^0-9a-zA-Z]+", "_", module_fqn).strip("_")


# Valid XNNPACK INT8 schemes for this export path (symmetric W, asymmetric acts).
# Static activation only: dynamic PT2E inserts quantized_decomposed::choose_qparams / quantize_per_tensor
# which often hit ExecuTorch ToOutVarPass "Missing out variants" on current torch+executorch builds.
# "none" = leave layer FP32 for PT2E.
XNNPACK_LAYER_QUANT_CHOICES: Tuple[str, ...] = (
    "none",
    "int8_pc_static",
    "int8_pt_static",
)


def scheme_to_xnnpack_config(scheme: str):
    if scheme == "none":
        raise ValueError("scheme 'none' must skip set_module_name")
    if scheme == "int8_pc_static":
        return get_symmetric_quantization_config(
            is_per_channel=True, is_dynamic=False
        )
    if scheme == "int8_pt_static":
        return get_symmetric_quantization_config(
            is_per_channel=False, is_dynamic=False
        )
    if scheme == "int8_pc_dynamic":
        return get_symmetric_quantization_config(
            is_per_channel=True, is_dynamic=True
        )
    if scheme == "int8_pt_dynamic":
        return get_symmetric_quantization_config(
            is_per_channel=False, is_dynamic=True
        )
    raise ValueError(f"Unknown scheme {scheme!r}")


# Coerce legacy Optuna "*_dynamic" labels when loading old trials / user_attrs.
LAYER_QUANT_SCHEME_EXPORT_MAP: Dict[str, str] = {
    "int8_pc_dynamic": "int8_pc_static",
    "int8_pt_dynamic": "int8_pt_static",
}


def resolve_layer_quant_scheme_for_export(scheme: str) -> str:
    return LAYER_QUANT_SCHEME_EXPORT_MAP.get(scheme, scheme)


EDGE_DEVICE_DATASET_PATH → /Users/souparna/AndroidStudioProjects/ADLSProject/mase/tiny-imagenet-200


### 1. Load Metrics from Android Phone

If you are trying to optimise and deploy the model on an Android phone, please plug in the phone to the laptop and make sure the developer mode is on. In settings make sure debugging mode is turned on. Run the following code cell to extract the hardware information. A JSON file will be created to hold the information and it will be used in the configuration setup.

In [6]:
hw = get_hardware_for_phone() # Make sure the phone is connected and ADB is set up
print("Optimizing for:", hw["model"])
print(hw["gpu"], hw["cpu_cores"], hw["ram_gb"])

shell getprop ro.product.manufacturer -> 'Google\n'...
shell getprop ro.product.model -> 'sdk_gphone64_arm64\n'...
shell getprop ro.product.brand -> 'google\n'...
shell getprop ro.product.soc_manufacturer -> '\n'...
shell getprop ro.soc_manufacturer -> '\n'...
shell getprop ro.product.soc_model -> '\n'...
shell getprop ro.soc_model -> '\n'...
shell getprop ro.product.hardware -> '\n'...
shell getprop ro.hardware -> 'ranchu\n'...
shell getprop ro.product.supported_abis -> '\n'...
shell getprop ro.supported_abis -> '\n'...
shell getprop ro.product.sdk_int -> '\n'...
shell getprop ro.sdk_int -> '\n'...
shell cat /proc/cpuinfo -> 'processor\t: 0\nBogoMIPS\t: 48.00\nFeatures\t: fp asimd evtstrm aes pmull sha1 sha2 crc32 atomics fphp asimdhp cpuid asimdrdm jscvt fcma lrcpc dcpop sha3 asimddp sha512 asimdfhm dit uscat ilrcpc flagm ssbs'...
shell cat /proc/meminfo | grep MemTotal -> 'MemTotal:        4014920 kB\n'...
shell dumpsys SurfaceFlinger | grep GLES -> ' ------------RE GLES (Ganesh)---

In [7]:
# # NPU/NNAPI hardware check (non-reference devices indicate accelerators like NPU)
# # Reload so notebook picks up latest ``get_adb_sh_c`` (e.g. ``timeout_sec``) without kernel restart.
# import importlib
# import EDGE_device.device_specifications as _adb_spec_mod
# importlib.reload(_adb_spec_mod)
# get_adb_output = _adb_spec_mod.get_adb_output
# get_adb_sh_c = _adb_spec_mod.get_adb_sh_c

# # ``get_adb_output`` / ``get_adb_sh_c`` use a timeout (default 20s, env ``ADB_SHELL_TIMEOUT_SEC``).
# # Avoid ``service list`` here — it often blocks for a long time on real devices.
# checks_simple = {
#     "ro.hardware": "shell getprop ro.hardware",
# }
# checks_shell = {
#     # ``grep -i npu`` false-positives on ``c2inputsurface`` (substring ``input`` → n-p-u)
#     "npu_prop": "getprop | grep -i npu | grep -vi c2inputsurface || true",
#     "cpuinfo": "head -20 /proc/cpuinfo",
#     "nn_hal": "ls /vendor/lib/hw/ /vendor/lib64/hw/ 2>/dev/null | grep neuralnetworks || echo none",
#     "init_neural_svcs": "getprop | grep -i init.svc.neural || true",
# }

# results = {k: get_adb_output(v) for k, v in checks_simple.items()}
# for k, script in checks_shell.items():
#     results[k] = get_adb_sh_c(script, timeout_sec=15.0)

# # Heuristic NPU detection
# npu_indicators = ["npu", "hexagon", "qcom", "kirin", "mali", "neuralnetworks"]
# has_npu = any(any(ind in res.lower() for ind in npu_indicators) for res in results.values())
# print(f"NPU/Accel likely: {'YES' if has_npu else 'NO'}")
# print("Full results:", results)

In [8]:
# specs = load_phone_specs('EDGE_device/device_specifications.json')          # uses default PHONE_SPECS_FILE
# print("Specs dict:", specs)
# print("Keys:", list(specs.keys()))

# if specs:
#     hw = list(specs.values())[0]
#     print(hw["model"], hw["gpu"], hw["cpu_cores"], hw["ram_gb"])
# else:
#     print("No specs found; JSON file missing or empty.")

### 2. Defining the Search Space

Here the code prepares the configurations for the pipeline

In [9]:
def edge_optuna_config(trial) -> Dict[str, Any]:
    """Per-layer XNNPACK-valid INT8 (or none) + L1 pruning amount."""

    prune_ratio = trial.suggest_float("prune_ratio", 0.0, 0.7)
    layer_quant: Dict[str, str] = {}
    for mod_name in QUANT_LAYER_NAMES:
        pname = optuna_param_name_for_module(mod_name)
        layer_quant[mod_name] = trial.suggest_categorical(
            pname, list(XNNPACK_LAYER_QUANT_CHOICES)
        )

    return {
        "prune_ratio": prune_ratio,
        "backend": "xnnpack",
        "use_mixed_delegation": False,
        "delegation_plan": None,
        "layer_quant": layer_quant,
    }



### 3. Model Optimisation

In [10]:
def _tiny_train_root() -> Path:
    for base in (
        PROJECT_ROOT / "mase" / "tiny-imagenet-200",
        PROJECT_ROOT / "tiny-imagenet-200",
    ):
        t = base / "train"
        if t.is_dir():
            return t
    return PROJECT_ROOT / "mase" / "tiny-imagenet-200" / "train"


def _load_resnet18_from_pt(pt_path: Path, load_device: str) -> nn.Module:
    """Load ResNet18 from .pt: dict+state_dict, raw state_dict, or full nn.Module (torch.save(model))."""
    if not pt_path.is_file():
        raise FileNotFoundError(
            f"Missing {pt_path!s}. Add resnet18_qat_fp32.pt under python notebooks/ (e.g. export from ResNet18.ipynb)."
        )
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    n_cls = 200
    if isinstance(ckpt, nn.Module):
        sd = ckpt.state_dict()
        if hasattr(ckpt, "fc") and isinstance(ckpt.fc, nn.Linear):
            n_cls = int(ckpt.fc.out_features)
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        inner = ckpt["state_dict"]
        n_cls = int(ckpt.get("num_classes", 200))
        if isinstance(inner, nn.Module):
            sd = inner.state_dict()
            if hasattr(inner, "fc") and isinstance(inner.fc, nn.Linear):
                n_cls = int(inner.fc.out_features)
        else:
            sd = inner
    elif isinstance(ckpt, dict):
        sd = ckpt
        if "fc.bias" in sd:
            n_cls = int(sd["fc.bias"].shape[0])
    else:
        raise TypeError(f"Unsupported checkpoint type {type(ckpt)}; expected dict or nn.Module.")
    if not isinstance(sd, dict):
        raise TypeError(f"Expected state_dict to be dict-like, got {type(sd)}.")
    model = models.resnet18(weights=None)
    model.fc = nn.Linear(model.fc.in_features, n_cls)
    model.load_state_dict(sd, strict=True)
    return model.to(load_device)


def _load_mobilenetv3_from_pt(pt_path: Path, load_device: str) -> nn.Module:
    """Load MobileNet V3 Large from .pt (torchvision layout: head is classifier[3], not fc)."""
    if not pt_path.is_file():
        raise FileNotFoundError(
            f"Missing {pt_path!s}. Add MobileNet-V3.pt under python notebooks/ (e.g. export from MobileNetV3.ipynb)."
        )
    ckpt = torch.load(pt_path, map_location="cpu", weights_only=False)
    n_cls = 200

    def _ncls_from_classifier_module(m: nn.Module) -> int | None:
        if hasattr(m, "classifier") and isinstance(m.classifier, nn.Sequential):
            last = m.classifier[-1]
            if isinstance(last, nn.Linear):
                return int(last.out_features)
        return None

    if isinstance(ckpt, nn.Module):
        sd = ckpt.state_dict()
        n = _ncls_from_classifier_module(ckpt)
        if n is not None:
            n_cls = n
        elif hasattr(ckpt, "fc") and isinstance(ckpt.fc, nn.Linear):
            n_cls = int(ckpt.fc.out_features)
    elif isinstance(ckpt, dict) and "state_dict" in ckpt:
        inner = ckpt["state_dict"]
        n_cls = int(ckpt.get("num_classes", n_cls))
        if isinstance(inner, nn.Module):
            sd = inner.state_dict()
            n = _ncls_from_classifier_module(inner)
            if n is not None:
                n_cls = n
            elif hasattr(inner, "fc") and isinstance(inner.fc, nn.Linear):
                n_cls = int(inner.fc.out_features)
        else:
            sd = inner
    elif isinstance(ckpt, dict):
        sd = ckpt
        if "classifier.3.bias" in sd:
            n_cls = int(sd["classifier.3.bias"].shape[0])
        elif "fc.bias" in sd:
            n_cls = int(sd["fc.bias"].shape[0])
    else:
        raise TypeError(f"Unsupported checkpoint type {type(ckpt)}; expected dict or nn.Module.")
    if not isinstance(sd, dict):
        raise TypeError(f"Expected state_dict to be dict-like, got {type(sd)}.")
    if "classifier.3.bias" in sd:
        n_cls = int(sd["classifier.3.bias"].shape[0])
    model = models.mobilenet_v3_large(weights=None)
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Linear(in_features, n_cls)
    model.load_state_dict(sd, strict=True)
    return model.to(load_device)


def _new_module_from_cpu_state_dict(sd: Dict[str, torch.Tensor]) -> nn.Module:
    """Fresh torchvision model matching checkpoint keys (ResNet18 vs MobileNet V3 Large)."""
    keys = set(sd.keys())
    if "classifier.3.weight" in keys or any(k.startswith("features.") for k in keys):
        m = models.mobilenet_v3_large(weights=None)
        n_cls = int(sd["classifier.3.bias"].shape[0])
        in_f = m.classifier[3].in_features
        m.classifier[3] = nn.Linear(in_f, n_cls)
        m.load_state_dict(sd, strict=True)
        return m
    if "fc.weight" in keys:
        m = models.resnet18(weights=None)
        n_cls = int(sd["fc.bias"].shape[0])
        m.fc = nn.Linear(m.fc.in_features, n_cls)
        m.load_state_dict(sd, strict=True)
        return m
    raise ValueError(
        "BASE_STATE_DICT is neither MobileNet V3 Large nor ResNet18; "
        f"sample keys: {list(sd.keys())[:10]}"
    )


def load_model_and_train() -> nn.Module:
    """Load base checkpoint once; each call returns a fresh FP32 module (same arch)."""
    global BASE_STATE_DICT
    if BASE_STATE_DICT is None:
        print(f"LOADING BASE WEIGHTS FROM {MOBILENET_V3_PATH.name}")
        load_device = (
            "mps"
            if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
            else ("cuda" if torch.cuda.is_available() else "cpu")
        )
        model = _load_mobilenetv3_from_pt(MOBILENET_V3_PATH, load_device)
        model.eval()
        print("Model is prepared.....")
        BASE_STATE_DICT = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    load_device = (
        "mps"
        if getattr(torch.backends, "mps", None) and torch.backends.mps.is_available()
        else ("cuda" if torch.cuda.is_available() else "cpu")
    )
    m = _new_module_from_cpu_state_dict(BASE_STATE_DICT)
    return m.to(load_device)


def apply_torch_l1_prune(model: nn.Module, amount: float) -> None:
    """Unstructured L1 prune on Conv2d / Linear weights; masks merged into .weight for export."""
    if amount <= 0:
        return
    to_prune: List[nn.Module] = [
        m for m in model.modules() if isinstance(m, (nn.Conv2d, nn.Linear))
    ]
    for m in to_prune:
        prune.l1_unstructured(m, name="weight", amount=amount)
    for m in to_prune:
        prune.remove(m, "weight")


def edge_optimise_model(
    config: Dict[str, Any],
    enable_qat: bool = False,
    model: Optional[nn.Module] = None,
) -> nn.Module:
    """Fresh FP32 model + optional torch pruning. PT2E INT8 happens in ``edge_export_model``."""
    del enable_qat  # QAT not used on this path; XNNPACK PT2E uses calibration at export.
    if model is None:
        model = load_model_and_train()
    model = model.cpu()
    apply_torch_l1_prune(model, float(config.get("prune_ratio", 0.0)))
    model.eval()
    return model


def edge_host_val_sanity_check(
    model: nn.Module,
    *,
    train_dir: Optional[Path] = None,
    max_batches: int = 10,
    batch_size: int = 64,
) -> Optional[float]:
    """Mean top-1 over a few Tiny ImageNet *train* ImageFolder batches (same class order as Android)."""
    root = train_dir or _tiny_train_root()
    if not root.is_dir():
        return None
    transform = T.Compose(
        [
            T.Resize(224),
            T.ToTensor(),
            T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
        ]
    )
    loader = torch.utils.data.DataLoader(
        datasets.ImageFolder(root=str(root), transform=transform),
        batch_size=batch_size,
        shuffle=True,
        num_workers=0,
    )
    model.eval()
    device = next(model.parameters()).device
    correct = total = 0
    with torch.no_grad():
        for i, (imgs, labels) in enumerate(loader):
            if i >= max_batches:
                break
            imgs, labels = imgs.to(device), labels.to(device)
            logits = model(imgs)
            pred = logits.argmax(dim=1)
            correct += int((pred == labels).sum().item())
            total += int(labels.numel())
    return (correct / total) if total else None



### 4. Export .pte file for Edge Device

In [11]:
def _calibration_input_batches(
    max_batches: int = 4,
    batch_size: int = 8,
) -> List[torch.Tensor]:
    """Representative inputs for PT2E observers; Tiny ImageNet train if present else random."""
    root = _tiny_train_root()
    if root.is_dir():
        transform = T.Compose(
            [
                T.Resize(224),
                T.ToTensor(),
                T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
            ]
        )
        loader = torch.utils.data.DataLoader(
            datasets.ImageFolder(root=str(root), transform=transform),
            batch_size=batch_size,
            shuffle=True,
            num_workers=0,
        )
        out: List[torch.Tensor] = []
        for i, (imgs, _) in enumerate(loader):
            if i >= max_batches:
                break
            out.append(imgs.cpu())
        if out:
            return out
    return [torch.randn(batch_size, 3, 224, 224)]


def edge_export_model(model: nn.Module, trial_id: int, config: Dict[str, Any]) -> str:
    """ExecuTorch .pte: XNNPACK PT2E (prepare_pt2e / convert_pt2e) + to_edge_transform_and_lower."""
    model.eval().cpu()
    sample_inputs = (torch.randn(1, 3, 224, 224),)
    layer_quant: Dict[str, str] = config.get("layer_quant") or {}
    any_quant = any(scheme != "none" for scheme in layer_quant.values())

    if any_quant:
        quantizer = XNNPACKQuantizer()
        for name, scheme in layer_quant.items():
            if scheme == "none":
                continue
            eff = resolve_layer_quant_scheme_for_export(scheme)
            if eff != scheme:
                warnings.warn(
                    f"Layer {name!r}: {scheme!r} -> {eff!r} for export "
                    "(dynamic activation PT2E ops are not lowered by this to_executorch path).",
                    UserWarning,
                    stacklevel=1,
                )
            quantizer.set_module_name(name, scheme_to_xnnpack_config(eff))

        ep = torch.export.export(model, sample_inputs)
        prepared = prepare_pt2e(ep.module(), quantizer)
        with torch.no_grad():
            for batch in _calibration_input_batches():
                for i in range(batch.shape[0]):
                    prepared(batch[i : i + 1])
        quantized_model = convert_pt2e(prepared)
        lowered = to_edge_transform_and_lower(
            torch.export.export(quantized_model, sample_inputs),
            partitioner=[XnnpackPartitioner()],
        )
    else:
        lowered = to_edge_transform_and_lower(
            torch.export.export(model, sample_inputs),
            partitioner=[XnnpackPartitioner()],
        )

    exec_prog = lowered.to_executorch()
    PTE_DIR = Path("pte_files")
    PTE_DIR.mkdir(exist_ok=True)
    pte_path = PTE_DIR / f"mobilenetv3_trial_{trial_id}.pte"
    with open(pte_path, "wb") as f:
        exec_prog.write_to_file(f)
    return str(pte_path)



In [12]:
import shlex

from EDGE_device.device_specifications import get_adb_path

def move_pte_to_app(pte_path: str, app_name: str):
    """Moves model to the app's private storage on the Android phone."""
    adb = get_adb_path()

    maybe_model_name = re.search(r'([^/\\]+\.pte)$', pte_path)
    model_name = maybe_model_name.group(1) if maybe_model_name else "model.pte"

    subprocess.run([adb, "push", pte_path, "/data/local/tmp/"], check=True, timeout=30,)
    subprocess.run([adb, "shell", "run-as", app_name, "cp", f"/data/local/tmp/{model_name}", "files/model.pte",], check=True, timeout=30,)
    
def move_dataset_to_app(dataset_path: str, app_name: str):
    """Copy dataset into files/dataset/ so MV3Demo sees files/dataset/train|val|test/ (ImageFolder)."""
    adb = get_adb_path()
    dataset_name = os.path.basename(os.path.normpath(dataset_path))
    subprocess.run([adb, "push", dataset_path, "/data/local/tmp/"], check=True, timeout=600)
    src_dot = shlex.quote(f"/data/local/tmp/{dataset_name}/.")
    inner_script = (
        f"DEST=$(ls -d /data/user/*/{app_name}/files 2>/dev/null | head -n1); "
        f'[ -n "$DEST" ] || DEST=/data/data/{app_name}/files; '
        f'test -d "$DEST" && rm -rf "$DEST/dataset" && mkdir -p "$DEST/dataset" && '
        f'cp -R {src_dot} "$DEST/dataset/"'
    )
    remote_line = f"run-as {shlex.quote(app_name)} sh -c {shlex.quote(inner_script)}"
    r = subprocess.run(
        [adb, "shell", remote_line],
        capture_output=True,
        text=True,
        timeout=600,
    )
    if r.returncode != 0:
        raise subprocess.CalledProcessError(r.returncode, [adb, "shell", remote_line], r.stdout, r.stderr)


def clear_edge_metrics_json_on_device(app_name: str) -> None:
    """Remove all metrics_trial_*.json under app files/ (avoids stale reads before a study or trial)."""
    adb = get_adb_path()
    inner_script = (
        f"DEST=$(ls -d /data/user/*/{app_name}/files 2>/dev/null | head -n1); "
        f'[ -n "$DEST" ] || DEST=/data/data/{app_name}/files; '
        f'rm -f "$DEST"/metrics_trial_*.json'
    )
    remote_line = f"run-as {shlex.quote(app_name)} sh -c {shlex.quote(inner_script)}"
    subprocess.run(
        [adb, "shell", remote_line],
        capture_output=True,
        text=True,
        timeout=30,
    )


### Benchmark for the trials on the phone

The functions push the optimised `.pte` and (once) a dataset folder into **Image Classification** (MV3Demo project) private storage (`files/model.pte`, `files/dataset/...`), then start **MainActivity** with action `com.image_classification_app.action.BENCHMARK`. **MaseOptimise** (in the app sources) runs ImageFolder eval and writes **`metrics_trial_<trial_id>.json`** into internal **`files/`** (same private storage as the model). The host polls with `adb exec-out run-as <app_id> cat files/metrics_trial_<id>.json` (requires a **debuggable** build so `run-as` works).

Pipeline: install the app once, set `EDGE_DEVICE_DATASET_PATH` (host folder with `train/`, `val/`, `test/`), set `EDGE_EVAL_SPLIT` if not `val`, then run the study.

Per trial: `adb push` .pte → `run-as` copy to `files/model.pte` → `am start` with extras `split`, `trial_id` → poll/pull metrics JSON.


In [13]:
from typing import Any

from EDGE_device.device_specifications import get_adb_path

def _metrics_payload_ready(data: dict) -> bool:
    st = data.get("status")
    if st in ("done", "error"):
        return True
    return isinstance(data, dict) and "latency_p95_ms" in data and "top1_acc" in data


def _adb_cat_stdout_not_metrics_json(raw: bytes) -> bool:
    """True when cat failed or file is missing — error text often on stdout with adb returncode 0."""
    if not (raw or b"").strip():
        return True
    if raw.lstrip().startswith(b"cat:"):
        return True
    if b"No such file or directory" in raw:
        return True
    if b"Permission denied" in raw:
        return True
    return False


def edge_benchmark_trial(
    trial_id: int,
    pte_path: str,
    dataset_path: Optional[str] = None,
    split: Optional[str] = None,
    app_name: str = IMAGE_CLASSIFICATION_APP_ID,
    *,
    poll_timeout_sec: Optional[int] = None,
    max_images: Optional[int] = None,
) -> dict[str, Any]:
    """Push → benchmark → metrics. MaseOptimise writes JSON to internal files/; host reads via run-as cat."""

    ds = dataset_path if dataset_path is not None else EDGE_DEVICE_DATASET_PATH
    ev_split = (split or EDGE_EVAL_SPLIT).lower()
    benchmark_action = f"{app_name}.action.BENCHMARK"
    cap = EDGE_BENCHMARK_MAX_IMAGES_FLAG if max_images is None else max_images
    try:
        move_pte_to_app(pte_path=pte_path, app_name=app_name)
        if trial_id == 0 and ds:
            move_dataset_to_app(dataset_path=ds, app_name=app_name)

        adb = get_adb_path()
        am_cmd = [
            adb,
            "shell",
            "am",
            "start",
            "-n",
            f"{app_name}/.MainActivity",
            "-a",
            benchmark_action,
            "-e",
            "split",
            ev_split,
            "-e",
            "trial_id",
            str(trial_id),
        ]
        if cap is not None and cap > 0:
            am_cmd.extend(["--ei", "max_images", str(cap)])
        # Drop stale JSON or the host reads the *previous* benchmark and returns immediately.
        clear_edge_metrics_json_on_device(app_name)
        subprocess.run(am_cmd, check=True, timeout=30)

        return edge_poll_metrics(
            trial_id,
            app_package=app_name,
            timeout=poll_timeout_sec
            if poll_timeout_sec is not None
            else EDGE_METRICS_POLL_TIMEOUT_FLAG,
        )
    except Exception as e:
        print(f"Trial {trial_id} failed: {e}")
        return {
            "top1_acc": 0.0,
            "latency_p95_ms": 999.0,
            "memory_peak_mb": 0.0,
            "num_samples": 0,
            "error": str(e),
            "status": "error",
            "valid_forward_count": 0,
            "decode_failures": 0,
        }


def edge_poll_metrics(
    trial_id: int,
    timeout: Optional[int] = None,
    app_package: str = IMAGE_CLASSIFICATION_APP_ID,
) -> dict[str, Any]:
    """Poll metrics JSON from app internal filesDir via adb (debuggable builds: run-as cat)."""

    timeout = EDGE_METRICS_POLL_TIMEOUT_FLAG if timeout is None else timeout
    remote_rel = f"files/metrics_trial_{trial_id}.json"
    adb = get_adb_path()
    start = time.time()
    last_info_ts = 0.0
    time.sleep(2)
    while time.time() - start < timeout:
        proc: Optional[subprocess.CompletedProcess[bytes]] = None
        raw_out = b""
        try:
            proc = subprocess.run(
                [adb, "exec-out", "run-as", app_package, "cat", remote_rel],
                capture_output=True,
                timeout=30,
            )
            raw_out = proc.stdout or b""
            stripped = raw_out.strip()
            if proc.returncode != 0:
                err_txt = (proc.stderr or b"").decode("utf-8", errors="replace").strip()
                if err_txt:
                    print(f"  [adb] cat rc={proc.returncode}: {err_txt[:400]}")
                time.sleep(2)
                continue
            if not stripped:
                time.sleep(2)
                continue
            if _adb_cat_stdout_not_metrics_json(raw_out):
                now = time.time()
                if now - last_info_ts >= 30:
                    last_info_ts = now
                    elapsed = int(now - start)
                    print(
                        f"  [poll] metrics file not written yet ({elapsed}s / {timeout}s). "
                        "MaseOptimise only creates JSON after the full eval loop finishes "
                        f"(lower EDGE_BENCHMARK_MAX_IMAGES_FLAG for a shorter run). remote={remote_rel!r}"
                    )
                time.sleep(2)
                continue
            try:
                text = stripped.decode("utf-8")
                data = json.loads(text)
            except json.JSONDecodeError as je:
                preview = repr(raw_out[:200])
                print(
                    f"  [poll] JSONDecodeError: {je!s}; rc={proc.returncode}; stdout_preview={preview}"
                )
                time.sleep(2)
                continue

            if not _metrics_payload_ready(data):
                time.sleep(2)
                continue

            err = data.get("error")
            if err and str(err).lower() not in ("null", "none", ""):
                print(f"  [metrics] device error field: {err}")

            st_raw = data.get("status")
            status_str = st_raw if isinstance(st_raw, str) else ""
            out: dict[str, Any] = {
                "top1_acc": float(data.get("top1_acc", 0.0)),
                "latency_p95_ms": float(data.get("latency_p95_ms", 999.0)),
                "memory_peak_mb": float(data.get("memory_peak_mb", 0.0)),
                "num_samples": int(data.get("num_samples", 0)),
                "status": status_str,
                "valid_forward_count": int(data.get("valid_forward_count", 0)),
                "decode_failures": int(data.get("decode_failures", 0)),
            }
            if data.get("num_samples_total_split") is not None:
                out["num_samples_total_split"] = int(data["num_samples_total_split"])
            out["error"] = None if err in (None, "null", "") else str(err)
            out["trial_id_json"] = data.get("trial_id")
            out["split_json"] = data.get("split")
            return out
        except subprocess.TimeoutExpired:
            print("  [poll] adb cat timed out")
            time.sleep(2)
        except Exception as ex:
            print(f"  [poll] {ex!r}")
            time.sleep(2)

    print(
        f"  [poll] timeout after {timeout}s — no readable {remote_rel}. "
        f"Use a debuggable build, or: adb shell run-as {app_package} ls files/"
    )
    return {
        "top1_acc": 0.0,
        "latency_p95_ms": 999.0,
        "memory_peak_mb": 9999.0,
        "num_samples": 0,
        "error": "poll_timeout",
        "status": "error",
        "valid_forward_count": 0,
        "decode_failures": 0,
    }


### 5. Defining the Objective Function

In [14]:
def objective(trial):
    """Full pipeline: config, model, export, phone, score."""

    config = edge_optuna_config(trial)
    
    model = edge_optimise_model(config, enable_qat=False)
    print("Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model")
    
    host_acc = edge_host_val_sanity_check(model)
    if host_acc is not None:
        print(f"  [host sanity] ~top1 over few train batches: {host_acc:.4f}")
        trial.set_user_attr("host_train_sanity_top1", host_acc)
        if host_acc < 0.01:
            print(
                "  [host sanity] WARN: very low — check data path, .pt checkpoint, or prune strength."
            )
    else:
        print("  [host sanity] skipped (Tiny ImageNet train folder missing)")

    pte_path = edge_export_model(model, trial.number, config)

    metrics = edge_benchmark_trial(
        trial.number, pte_path, split=EDGE_EVAL_SPLIT
    )

    acc = float(metrics.get("top1_acc", 0.0))
    latency = float(metrics.get("latency_p95_ms", 999.0))
    memory = float(metrics.get("memory_peak_mb", 9999.0))
    n_samp = int(metrics.get("num_samples", 0))
    v_fwd = int(metrics.get("valid_forward_count", 0))
    dec_fail = int(metrics.get("decode_failures", 0))
    m_err = metrics.get("error")
    m_stat = metrics.get("status", "")

    score = acc - 0.02 * (latency / 30.0) - 0.001 * (memory / 2000.0)

    trial.set_user_attr("top1_acc", acc)
    trial.set_user_attr("latency_ms", latency)
    trial.set_user_attr("memory_mb", memory)
    trial.set_user_attr("num_samples", n_samp)
    trial.set_user_attr("valid_forward_count", v_fwd)
    trial.set_user_attr("decode_failures", dec_fail)
    trial.set_user_attr("metrics_status", m_stat)
    if m_err:
        trial.set_user_attr("metrics_error", m_err)
    for k, v in config.items():
        trial.set_user_attr(k, v)

    print(
        f"  → acc={acc:.3f}, lat={latency:.1f}ms, score={score:.3f} | "
        f"samples={n_samp}, forwards={v_fwd}, decode_fail={dec_fail}, status={m_stat!r}"
    )
    if m_err:
        print(f"  → metrics error field: {m_err}")
    return score


### 6. Optuna Study

In [ ]:
# Create + run study
# If you shrink XNNPACK_LAYER_QUANT_CHOICES, SQLite-backed studies may error on suggest_categorical;
# use a new study_name (e.g. mv3_edge_opt_v2) or delete optuna_mv3.db.
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=42),
    study_name="mv3_edge_opt",
    storage="sqlite:///optuna_mv3.db",  # Save progress
    load_if_exists=True,
)

# Run 50 trials
study.optimize(objective, n_trials=20)

print(f"\nBest trial #{study.best_trial.number}")
print(f"   Score: {study.best_value:.3f}")
print("   Config:", study.best_params)

# Quick viz

optuna.visualization.plot_optimization_history(study).show()


[I 2026-03-27 12:10:00,785] Using an existing study with name 'mv3_edge_opt' instead of creating a new one.


Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model
  [host sanity] ~top1 over few train batches: 0.7828
pte_files/mobilenetv3_trial_20.pte: 1 file pushed, 0 skipped. 26.2 MB/s (13407008 bytes in 0.487s)
Starting: Intent { act=com.image_classification_app.action.BENCHMARK cmp=com.image_classification_app/.MainActivity (has extras) }
  [poll] metrics file not written yet (2s / 3600s). MaseOptimise only creates JSON after the full eval loop finishes (lower EDGE_BENCHMARK_MAX_IMAGES_FLAG for a shorter run). remote='files/metrics_trial_20.json'


[I 2026-03-27 12:11:34,800] Trial 20 finished with value: 0.5809551113333332 and parameters: {'prune_ratio': 0.19196431900158153, 'q_features_0_0': 'int8_pc_static', 'q_features_1_block_0_0': 'int8_pc_static', 'q_features_1_block_1_0': 'int8_pc_static', 'q_features_2_block_0_0': 'int8_pc_static', 'q_features_2_block_1_0': 'none', 'q_features_2_block_2_0': 'int8_pc_static', 'q_features_3_block_0_0': 'int8_pc_static', 'q_features_3_block_1_0': 'int8_pc_static', 'q_features_3_block_2_0': 'int8_pt_static', 'q_features_4_block_0_0': 'int8_pc_static', 'q_features_4_block_1_0': 'int8_pc_static', 'q_features_4_block_2_fc1': 'none', 'q_features_4_block_2_fc2': 'int8_pc_static', 'q_features_4_block_3_0': 'none', 'q_features_5_block_0_0': 'none', 'q_features_5_block_1_0': 'int8_pc_static', 'q_features_5_block_2_fc1': 'int8_pt_static', 'q_features_5_block_2_fc2': 'int8_pt_static', 'q_features_5_block_3_0': 'int8_pc_static', 'q_features_6_block_0_0': 'int8_pc_static', 'q_features_6_block_1_0': 'int

  → acc=0.590, lat=13.6ms, score=0.581 | samples=100, forwards=100, decode_fail=0, status='done'
Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model
  [host sanity] ~top1 over few train batches: 0.9703
pte_files/mobilenetv3_trial_21.pte: 1 file pushed, 0 skipped. 167.4 MB/s (13407008 bytes in 0.076s)
Starting: Intent { act=com.image_classification_app.action.BENCHMARK cmp=com.image_classification_app/.MainActivity (has extras) }


[I 2026-03-27 12:13:09,298] Trial 21 finished with value: 0.7249494719999999 and parameters: {'prune_ratio': 0.00391959700374047, 'q_features_0_0': 'int8_pc_static', 'q_features_1_block_0_0': 'int8_pc_static', 'q_features_1_block_1_0': 'int8_pc_static', 'q_features_2_block_0_0': 'int8_pc_static', 'q_features_2_block_1_0': 'none', 'q_features_2_block_2_0': 'int8_pc_static', 'q_features_3_block_0_0': 'int8_pt_static', 'q_features_3_block_1_0': 'int8_pc_static', 'q_features_3_block_2_0': 'int8_pt_static', 'q_features_4_block_0_0': 'int8_pc_static', 'q_features_4_block_1_0': 'int8_pc_static', 'q_features_4_block_2_fc1': 'none', 'q_features_4_block_2_fc2': 'int8_pc_static', 'q_features_4_block_3_0': 'int8_pc_static', 'q_features_5_block_0_0': 'none', 'q_features_5_block_1_0': 'int8_pc_static', 'q_features_5_block_2_fc1': 'int8_pt_static', 'q_features_5_block_2_fc2': 'int8_pc_static', 'q_features_5_block_3_0': 'int8_pc_static', 'q_features_6_block_0_0': 'int8_pc_static', 'q_features_6_block_

...

  → acc=0.000, lat=15.2ms, score=-0.010 | samples=100, forwards=100, decode_fail=0, status='done'
Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model
  [host sanity] ~top1 over few train batches: 0.9750
pte_files/mobilenetv3_trial_31.pte: 1 file pushed, 0 skipped. 45.5 MB/s (13407008 bytes in 0.281s)
Starting: Intent { act=com.image_classification_app.action.BENCHMARK cmp=com.image_classification_app/.MainActivity (has extras) }
  [poll] metrics file not written yet (2s / 3600s). MaseOptimise only creates JSON after the full eval loop finishes (lower EDGE_BENCHMARK_MAX_IMAGES_FLAG for a shorter run). remote='files/metrics_trial_31.json'


[I 2026-03-27 12:33:49,873] Trial 31 finished with value: 0.7336743059999999 and parameters: {'prune_ratio': 0.0004266861555096632, 'q_features_0_0': 'int8_pc_static', 'q_features_1_block_0_0': 'int8_pc_static', 'q_features_1_block_1_0': 'int8_pc_static', 'q_features_2_block_0_0': 'int8_pc_static', 'q_features_2_block_1_0': 'none', 'q_features_2_block_2_0': 'int8_pc_static', 'q_features_3_block_0_0': 'int8_pt_static', 'q_features_3_block_1_0': 'int8_pc_static', 'q_features_3_block_2_0': 'int8_pt_static', 'q_features_4_block_0_0': 'int8_pc_static', 'q_features_4_block_1_0': 'int8_pc_static', 'q_features_4_block_2_fc1': 'none', 'q_features_4_block_2_fc2': 'int8_pc_static', 'q_features_4_block_3_0': 'int8_pc_static', 'q_features_5_block_0_0': 'none', 'q_features_5_block_1_0': 'int8_pc_static', 'q_features_5_block_2_fc1': 'int8_pt_static', 'q_features_5_block_2_fc2': 'int8_pc_static', 'q_features_5_block_3_0': 'int8_pc_static', 'q_features_6_block_0_0': 'int8_pc_static', 'q_features_6_bloc

  → acc=0.740, lat=9.5ms, score=0.734 | samples=100, forwards=100, decode_fail=0, status='done'
Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model
  [host sanity] ~top1 over few train batches: 0.9656
pte_files/mobilenetv3_trial_32.pte: 1 file pushed, 0 skipped. 51.0 MB/s (13407008 bytes in 0.251s)
Starting: Intent { act=com.image_classification_app.action.BENCHMARK cmp=com.image_classification_app/.MainActivity (has extras) }
  [poll] metrics file not written yet (2s / 3600s). MaseOptimise only creates JSON after the full eval loop finishes (lower EDGE_BENCHMARK_MAX_IMAGES_FLAG for a shorter run). remote='files/metrics_trial_32.json'


[I 2026-03-27 12:35:54,164] Trial 32 finished with value: 0.7239151666666667 and parameters: {'prune_ratio': 0.045035257604387535, 'q_features_0_0': 'int8_pc_static', 'q_features_1_block_0_0': 'int8_pc_static', 'q_features_1_block_1_0': 'int8_pc_static', 'q_features_2_block_0_0': 'int8_pc_static', 'q_features_2_block_1_0': 'none', 'q_features_2_block_2_0': 'int8_pc_static', 'q_features_3_block_0_0': 'int8_pt_static', 'q_features_3_block_1_0': 'int8_pc_static', 'q_features_3_block_2_0': 'int8_pt_static', 'q_features_4_block_0_0': 'int8_pc_static', 'q_features_4_block_1_0': 'int8_pc_static', 'q_features_4_block_2_fc1': 'none', 'q_features_4_block_2_fc2': 'int8_pc_static', 'q_features_4_block_3_0': 'int8_pc_static', 'q_features_5_block_0_0': 'none', 'q_features_5_block_1_0': 'int8_pc_static', 'q_features_5_block_2_fc1': 'int8_pt_static', 'q_features_5_block_2_fc2': 'int8_pc_static', 'q_features_5_block_3_0': 'int8_pc_static', 'q_features_6_block_0_0': 'int8_pc_static', 'q_features_6_block

  → acc=0.730, lat=9.1ms, score=0.724 | samples=100, forwards=100, decode_fail=0, status='done'
Model is pruned (FP32); XNNPACK PT2E quant runs in edge_export_model


In [16]:
optuna.visualization.plot_optimization_history(study).show()

### Extract the best model

In [17]:
# Get best config (user_attrs include full layer_quant dict from the trial)
best_trial = study.best_trial
best_config = {
    k: v
    for k, v in best_trial.user_attrs.items()
    if k
    in (
        "prune_ratio",
        "backend",
        "use_mixed_delegation",
        "delegation_plan",
        "layer_quant",
    )
}
# Fallback if attrs missing (e.g. old study): rebuild layer_quant from params
if "layer_quant" not in best_config and best_trial.params:
    best_config["prune_ratio"] = best_trial.params.get(
        "prune_ratio", best_config.get("prune_ratio", 0.0)
    )
    best_config["backend"] = "xnnpack"
    best_config["use_mixed_delegation"] = False
    best_config["delegation_plan"] = None
    best_config["layer_quant"] = {
        mod: best_trial.params[optuna_param_name_for_module(mod)]
        for mod in QUANT_LAYER_NAMES
    }

best_model = edge_optimise_model(best_config, enable_qat=False)
final_pte = edge_export_model(best_model, 999, best_config)

print(f"Best model: {final_pte}")
print("Deploy with: adb push", final_pte, "/sdcard/mobilenetv3_best.pte")



Best model: pte_files/mobilenetv3_trial_999.pte
Deploy with: adb push pte_files/mobilenetv3_trial_999.pte /sdcard/mobilenetv3_best.pte
